### MultiVectorRetriever

#### **개념:** 
* 여러 벡터 표현(예: 요약)과 원본 문서(큰 청크)를 함께 사용하여 검색의 정확성과 문맥적 이해를 높이는 검색 기법입니다.
#### **작동 방식:**
1.  **요약 생성:** 원본 문서를 작은 청크로 분할하고, 각 청크에 대한 요약을 생성합니다.
2.  **벡터 저장소:** 생성된 요약 문서를 임베딩하여 벡터 저장소(예: FAISS)에 인덱싱합니다.
3.  **문서 저장소:** 원본 문서 청크는 별도의 문서 저장소(예: LocalFileStore)에 저장되며, 요약 문서와 `doc_id`를 통해 연결됩니다.
4.  **검색 과정:**
    * 질의가 들어오면 먼저 벡터 저장소에서 질의와 유사한 요약 문서를 검색합니다.
    * 검색된 요약 문서의 `doc_id`를 활용하여 해당 `doc_id`와 연결된 원본 문서 청크를 문서 저장소에서 검색합니다.
    * 최종적으로 원본 문서 청크를 반환하여 풍부한 문맥 정보를 제공합니다.
#### **주요 이점:**
* **정확성 향상:** 요약을 통해 의미적으로 유사한 문서를 더 효과적으로 찾아내어 검색 정확도를 높입니다.
* **문맥 보존:** 요약 검색 후 원본 문서를 제공함으로써, 답변 생성 시 더 넓은 문맥 정보를 활용할 수 있도록 합니다.
* **효율성:** 짧은 요약을 사용하여 벡터 검색을 수행하므로, 검색 속도를 향상시킬 수 있습니다.
#### **활용 분야:** 
* 전문적이고 복잡한 도메인에서 정확하고 상세한 정보 검색이 필요한 경우에 유용하게 활용됩니다.

In [9]:
from langchain_community.document_loaders import PyPDFLoader

# 문서 로드
loader = PyPDFLoader('../data/KCI_FI003153549_p5.pdf')
documents = loader.load()

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
)
chunks = text_splitter.split_documents(documents)

In [11]:
len(chunks)

4

In [14]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
import uuid

# 문서 요약
prompt_text = '다음 문서의 요약을 생성하세요:\n\n{doc}'

prompt = ChatPromptTemplate.from_template(prompt_text)

# 도커를 이용하고 있으므로 base_url을 지정해주어야 함
llm = OllamaLLM(
    model="gemma3:270m",
    base_url="http://localhost:11434")
    
summarize_chain = {
    'doc': lambda x: x.page_content} | prompt | llm | StrOutputParser()

summaries = summarize_chain.batch(chunks, {'max_concurrency': 5}) # 최대 5개의 작업을 동시에 실행하도록 제한하는 설정

id_key = 'doc_id'

# 문서와 동일한 길이가 필요하므로 summaries에서 chunks로 변경
doc_ids = [str(uuid.uuid4()) for _ in chunks]

# 각 요약은 doc_id를 통해 원본 문서와 연결
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

In [15]:
summary_docs[1]

Document(metadata={'doc_id': '290a0cf6-34ea-4f0a-a4cb-1263892cda98'}, page_content='## 문서 요약\n\n본 연구는 의료기기 임상시험에 LLM 기반 AI를 활용하여 데이터 분석 및 의사 결정에 필요한 정확성과 효율성을 향상시키는 것을 목표로 한다. 이 접근 방식은 도메인 특화 데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로 구성된다. 각 단계는 의료기기 임상시험 분야 특성을 반영하여 상호 유기적으로 작동하며, Figure 1과 같이 이를 통해 해당 분야에서 최적의 성능을 달성하도록 설계된다.\n\n## 연구 목표\n\n본 연구는 의료기기 임상시험에 LLM 기반 AI를 활용하여 데이터 분석 및 의사 결정에 필요한 정확성과 효율성을 향상시키는 것을 목표로 한다.\n\n## 연구 범위\n\n본 연구는 의료기기 임상시험에서 LLM 기반 AI를 활용하여 데이터 분석 및 의사 결정에 필요한 정확성과 효율성을 향상시키는 것을 목표로 한다.\n\n## 연구 방법론\n\n본 연구는 의료기기 임상시험에 LLM 기반 AI를 활용하여 데이터 분석 및 의사 결정에 필요한 정확성과 효율성을 향상시키는 것을 목표로 한다.\n\n## 연구 결과\n\n본 연구 결과는 의료기기 임상시험에서 LLM 기반 AI를 활용하여 데이터 분석 및 의사 결정에 필요한 정확성과 효율성을 향상시키는 것을 목표로 한다.\n\n## 연구의 한계\n\n본 연구는 의료기기 임상시험에서 LLM 기반 AI를 활용하여 데이터 분석 및 의사 결정에 필요한 정확성과 효율성을 향상시키는 것을 목표로 한다.\n\n## 연구의 기대 효과\n\n본 연구는 의료기기 임상시험에서 LLM 기반 AI를 활용하여 데이터 분석 및 의사 결정에 필요한 정확성과 효율성을 향상시키는 것을 목표로 한다.\n\n## 연구의 향후 방향\n\n본 연구는 의료기기 임상시험에서 LLM 기반 AI를 활용하여 데이터 분석 및 의

In [16]:
from langchain_ollama import OllamaEmbeddings

# * bge-m3 임베딩 모델 준비
# OllamaEmbeddings 랭체인 문서: https://python.langchain.com/docs/integrations/text_embedding/ollama/
embedding_model = OllamaEmbeddings(
    model="bge-m3",
)

In [17]:
from langchain_community.vectorstores import FAISS

# 벡터 저장소(FAISS)에 요약을 인덱싱
vectorstore = FAISS.from_documents(summary_docs, embedding_model)

In [18]:
# FAISS 벡터 저장소를 로컬 파일에 저장
vectorstore.save_local("./faiss_index")

In [21]:
from langchain_classic.storage import LocalFileStore, create_kv_docstore

# 원문은 별도 docstore 폴더로 관리
byte_store = LocalFileStore("./docstore")
store = create_kv_docstore(byte_store)

In [22]:
# 원본 문서를 문서 저장소에 저장 (doc_id로 연결)
# 문서 내용은 유니코드 형식(\u)로 저장
store.mset(list(zip(doc_ids, chunks)))

In [23]:
from langchain.retrievers.multi_vector import MultiVectorRetriever

# MultiVectorRetriever 구성
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

ModuleNotFoundError: No module named 'langchain.retrievers'

In [14]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

#### 요약 검색

In [15]:
# 벡터 저장소가 요약을 검색
sub_docs = retriever.vectorstore.similarity_search(query, k=2)

In [16]:
sub_docs

[Document(id='3cb876e5-02b5-49a8-a862-96d011c208f4', metadata={'doc_id': 'c6091f4c-a2e9-400d-91ed-6e11f601ebe1'}, page_content='이 문서는 의료기기 임상시험 분야에 특화된 Private LLM 구축을 위한 데이터셋 구축 과정과 그 타당성에 대해 설명합니다. 총 11,954 페이지(111,954 페이지)의 의료기기 임상시험 관련 문서를 158개 수집했으며, 다음과 같이 분류되었습니다.\n\n*   **규제 문서 (30%):** FDA, EMA, PMDA 가이드라인, GCP 문서 등\n*   **교육 자료 (20%):** 임상시험 수행자 교육 매뉴얼, 온라인 강의 자료 등\n*   **프로토콜 및 보고서 (25%):** 임상시험 프로토콜, CSR 템플릿 등\n*   **의료기기 특화 문서 (15%):** 의료기기 임상시험 계획서, 기술 문서 등\n*   **기타 (10%):** 윤리위원회 관련 문서, 환자 동의서 템플릿 등\n\n수집된 데이터셋은 도메인 적합성, 다양성, 응용 가능성 측면에서 높은 타당성을 가지며, 임상시험의 규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포괄하는 것으로 평가됩니다.\n\n---\n\n**핵심 요약:**\n\n11,954 페이지의 의료기기 임상시험 관련 문서를 수집하여 Private LLM 구축을 위한 데이터셋을 구축했으며, 높은 타당성을 갖춘 것으로 평가되었습니다.'),
 Document(id='a202d4d8-4a91-4d4d-a952-94f6386cfcb6', metadata={'doc_id': 'aba6965c-3e80-43c9-8f27-e8bc35bc1d66'}, page_content='이 문서는 의료기기 임상시험 분야에 특화된 Private LLM 접근 방법을 제안합니다. 이 방법은 다음과 같은 네 단계로 구성됩니다.\n\n1. **도메인 특화 데이터셋 구축:** 의료기기 임상시험 전문가로부터 158개의 문서(총

In [17]:
print(sub_docs[0].page_content)

이 문서는 의료기기 임상시험 분야에 특화된 Private LLM 구축을 위한 데이터셋 구축 과정과 그 타당성에 대해 설명합니다. 총 11,954 페이지(111,954 페이지)의 의료기기 임상시험 관련 문서를 158개 수집했으며, 다음과 같이 분류되었습니다.

*   **규제 문서 (30%):** FDA, EMA, PMDA 가이드라인, GCP 문서 등
*   **교육 자료 (20%):** 임상시험 수행자 교육 매뉴얼, 온라인 강의 자료 등
*   **프로토콜 및 보고서 (25%):** 임상시험 프로토콜, CSR 템플릿 등
*   **의료기기 특화 문서 (15%):** 의료기기 임상시험 계획서, 기술 문서 등
*   **기타 (10%):** 윤리위원회 관련 문서, 환자 동의서 템플릿 등

수집된 데이터셋은 도메인 적합성, 다양성, 응용 가능성 측면에서 높은 타당성을 가지며, 임상시험의 규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포괄하는 것으로 평가됩니다.

---

**핵심 요약:**

11,954 페이지의 의료기기 임상시험 관련 문서를 수집하여 Private LLM 구축을 위한 데이터셋을 구축했으며, 높은 타당성을 갖춘 것으로 평가되었습니다.


#### 요약 검색 후 원본 문서 검색

**검색 과정**

1. 벡터 저장소(FAISS)에 질의를 던져 유사한 요약 문서 찾음

2. 찾아낸 요약 문서의 메타데이터에 있는 doc_id 추출

3. 추출된 doc_id를 사용하여 문서 저장소(docstore)에서 원본 문서 검색

4. 최종적으로 원본 문서 청크 반환

In [25]:
# 더 큰 원본 문서 청크를 반환
retrieved_docs = retriever.invoke(query)

In [26]:
len(retrieved_docs)

4

In [24]:
retrieved_docs

[Document(metadata={'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-27T02:09:00+09:00', 'moddate': '2024-12-27T02:09:00+09:00', 'source': '../data/KCI_FI003153549.pdf', 'total_pages': 12, 'page': 4, 'page_label': '5'}, page_content='의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하\n기 위해 의료기기 임상시험 전문가로부터 총 158개의 문\n서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음\n과 같이 분류된다:\n\x9f 규제 문서 (30%): FDA, EMA, PMDA 가이드라인, \nGCP 문서 등\n\x9f 교육 자료 (20%): 임상시험 수행자 교육 매뉴얼, 온라\n인 강의 자료 등\n\x9f 프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR \n(Clinical Study Report) 템플릿 등\n\x9f 의료기기 특화 문서 (15%): 의료기기 임상시험 계획\n서, 기술문서 등\n\x9f 기타 (10%): 윤리위원회 관련 문서, 환자 동의서 템플\n릿 등\n1.2 Validity of Collected Data\n수집된 데이터셋은 의료기기 임상시험에 특화된 \nPrivate LLM 구축을 위해 도메인 적합성과 다양성, 그리\n고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총 \n111,954페이지로 구성된 데이터는 의료기기 임상시험의 \n규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포\n괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수 \n있는 다양한 시나리오를 반영하도록 설계되었다.\n수집된 데이터는 규제 문서(30%), 교육 자료(20%), 프\n로토콜 및 보고서(25%), 의료기기 특화 문서(15%), 기타'),
 Document(metadata={'pro

In [21]:
retrieved_docs[0].page_content

'의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하\n기 위해 의료기기 임상시험 전문가로부터 총 158개의 문\n서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음\n과 같이 분류된다:\n\x9f 규제 문서 (30%): FDA, EMA, PMDA 가이드라인, \nGCP 문서 등\n\x9f 교육 자료 (20%): 임상시험 수행자 교육 매뉴얼, 온라\n인 강의 자료 등\n\x9f 프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR \n(Clinical Study Report) 템플릿 등\n\x9f 의료기기 특화 문서 (15%): 의료기기 임상시험 계획\n서, 기술문서 등\n\x9f 기타 (10%): 윤리위원회 관련 문서, 환자 동의서 템플\n릿 등\n1.2 Validity of Collected Data\n수집된 데이터셋은 의료기기 임상시험에 특화된 \nPrivate LLM 구축을 위해 도메인 적합성과 다양성, 그리\n고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총 \n111,954페이지로 구성된 데이터는 의료기기 임상시험의 \n규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포\n괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수 \n있는 다양한 시나리오를 반영하도록 설계되었다.\n수집된 데이터는 규제 문서(30%), 교육 자료(20%), 프\n로토콜 및 보고서(25%), 의료기기 특화 문서(15%), 기타'